# Estructura recomendada para modelos DES

Hay muchas formas de estructurar modelos en **SimPy**.

Cada programador tiene su propio enfoque. aqui Vamos a estructurar nuestro modelo de SimPy de manera **orientada a objetos**.

> Hay dos formas principales de programar en **Python**: **funcional** y **orientada a objetos**. A esto se le conoce como **paradigmas** de programación.

En la programación funcional, se escriben **funciones** que (idealmente) están diseñadas para hacer una sola cosa y hacerla bien.

Luego usas esas funciones en algún tipo de secuencia para lograr lo que intentas hacer.

Esto puede tener mucho sentido para muchos flujos de trabajo de análisis de datos y ciencia de datos.

En comparación, en la programación orientada a objetos (POO) todo se centra en objetos que tienen sus propios:

* **Atributos** (variables que les pertenecen), y
* **Métodos** (que es como se llaman las funciones cuando pertenecen a objetos).

Los **objetos** se crean como **instancias** de **clases**.

Las clases son, esencialmente, planos generalizados de cómo deberían funcionar ciertos tipos de objetos.

Esto puede ser muy útil en situaciones en las que tienes:

* Mucha lógica que tiene sentido adjuntar y organizar con la cosa que usa esa lógica (como un proceso), o
* La necesidad de hacer copias de una cosa muy similar (como muchos pacientes para poblar un modelo).

Veamos un ejemplo.

Supongamos que queremos escribir código que defina cómo funciona una ambulancia.

Habrá propiedades que la ambulancia tiene. Cosas como:

* La organización a la que pertenece.
* El número de matrícula.
* Si actualmente hay un paciente a bordo.
* Si la sirena está sonando en este momento.

También habrá cosas que la ambulancia hace:

* Conducir.
* Estacionar.
* Cargar y descargar pacientes.
* Encender y apagar la sirena.

En la Programación Orientada a Objetos, las cosas que **tiene** se conocen como **atributos** y las cosas que *hace* se llaman **métodos**.

Específicamente, tendremos 4 clases diferentes: **global**, **entity**, **model** y **trial**.

Aunque el código orientado a objetos puede sentirse un poco pesado para modelos pequeños, seguir el mismo tipo de patrón cada vez hace que sea muy fácil llevar el control de lo que está haciendo tu modelo. También hace mucho más sencillo **ampliar y modificar** el modelo con el tiempo.


## Clase global (glo)

La clase **global** almacena los valores de parámetros globales del modelo para que puedan cambiar aspectos del modelo y probar escenarios. Esto incluye:

- Valores para definir las distribuciones de **tiempos entre llegadas** (p. ej., media, desviación estándar, etc.).
- Valores para definir las **distribuciones de tiempos de actividad** (p. ej., media, desviación estándar, etc.).
- **Número** de cada recurso.
- **Duración** de las corridas de simulación.
- **Número de corridas** en un experimento (*trial*).

No creamos una instancia de **global**. En su lugar, la referenciamos directamente cuando necesitamos acceder a algo dentro de ella.

### Ejemplo


In [1]:
class glo:
    time_units_between_customer_arrivals = 5
    mean_customer_service_time = 6
    number_of_customer_support_agents = 1
    sim_duration = 1440
    number_of_runs = 10

## Clase entity

La clase de **entity** representa la entidad en el modelo; sigamos con el ejemplo en salud, donde a menudo será el/la **paciente**.

Aquí podemos almacenar los **atributos** que las entidades llevan consigo y alos que luego queremos acceder (piensa en una persona que porta una carpeta con información).

En un modelo sencillo, una entidad puede llevar solo su **cedula** y cuánto tiempo han estado esperando en cola. En modelos más complejos, también podrían incluir su **condición**, su **prioridad**, la **probabilidad** de seguir en un camino dado, etc.

### Ejemplo


In [2]:
class Customer:
    def __init__(self, p_id):
        self.id = p_id
        self.queue_time_customer_support_agent = 0

## Clase Model

La clase **Model** representa el sistema que estamos modelando, esto puede ser un servicio telefónico, un departamento clínico, etc. 

### miremos sus partes:

Primero, veremos el **constructor** de nuestro modelo. El cual configura:

- Un **entorno de SimPy** (básicamente donde vive todo).
- Un **contador de entidades** (que usaremos para asignar a las entidades, como pacientes, una cedula por ejemplo).
- Los **recursos** que necesitamos (por ejemplo, enfermeras).
- Un **DataFrame** para almacenar los resultados **por entidad** en una sola corrida del modelo.
- **Atributos** para guardar cosas como cuánto tiempo esperaron las entidades en cada actividad.

Lo que configura el constructor no tiene por qué limitarse a estas cosas; puede incluirse aquí cualquier elemento relacionado con el sistema en su conjunto que tenga sentido almacenar.

#### **Generador DES**: llegadas de entidades al sistema

Dentro de la clase **Model** tenemos una **función generadora** que representará nuestro generador DES para las entidades que **llegan** a nuestro proceso.

Básicamente funciona así:

*REPETIR LO SIGUIENTE INDEFINIDAMENTE (hasta que la simulación termine):*

1. Incrementa el contador para obtener el ID de la siguiente entidad (cedula de paciente).
2. Crea una nueva entidad y asígnale ese ID.
3. Inicia una instancia de la función generadora para su recorrido por el proceso e incorpórala en ella.
4. Muestrea el tiempo hasta la llegada de la siguiente entidad.
5. CONGELA esta función hasta que transcurra ese tiempo.
6. Vuelve al paso 1.

#### **Generador DES**: el recorrido de la entidad

Ahora veamos la parte principal. La otra función generadora: la que representa el recorrido de una entidad por el sistema (es donde enviamos las nuevas entidades generadas por el generador anterior).

Así funciona:

1. Registra el momento en que empieza a hacer fila (cola) para la primera actividad.
2. Solicita el recurso para la primera actividad.
3. Espera hasta que el recurso esté libre.
4. Cuando el recurso esté disponible, adquiérelo y mantenlo hasta terminar de usarlo. Registra el momento en que finaliza la espera y calcula el tiempo en cola.
5. Muestrea cuánto tiempo pasará en esta actividad.
6. CONGELA esta instancia de la función hasta que transcurra ese tiempo (congelando también el recurso, de modo que no esté disponible para nadie más).
7. Si hay otra actividad, repite lo mismo para ella. Si no, termina (y por tanto la entidad sale del modelo).


#### **Ejecución del modelo**

Creamos un método `run` en la clase **Model**. Básicamente, este método:

- **Inicializa** nuestros generadores DES (puntos de llegada) (aquí solo tenemos uno).
- Indica a la simulación que se ejecute por la **duración** especificada en la clase **global**.
- Llama al método que **calcula** los resultados de la corrida.
- **Imprime** el número de corrida junto con los resultados a nivel de paciente de esa corrida.

### ejemplo completo


In [3]:
class Model:
    # El constructor para preparar el modelo para una corrida. Pasamos un número de
    # corrida cuando creamos un nuevo modelo.
    def __init__(self, run_number):
        # Crear un entorno de SimPy en el que vivirá todo
        self.env = simpy.Environment()

        # Crear un contador de clientes (que usaremos como ID de cliente)
        self.customer_counter = 0

        # Crear un recurso de SimPy para representar a un agente de soporte al cliente,
        # que vivirá en el entorno creado arriba. La cantidad de este recurso
        # está dada por la capacidad, y tomamos ese valor de nuestra clase glo.
        self.customer_support_agent = simpy.Resource(self.env, capacity=glo.number_of_customer_support_agents)

        # Guardar el número de corrida recibido
        self.run_number = run_number

        # Crear un nuevo DataFrame de Pandas que almacenará algunos resultados
        # por ID de cliente (que usaremos como índice).
        self.results_df = pd.DataFrame()
        self.results_df["Customer ID"] = [1]
        self.results_df["Queue Time"] = [0.0]
        self.results_df["Time with Customer Support Agent"] = [0.0]
        self.results_df.set_index("Customer ID", inplace=True)

        # Crear un atributo para almacenar el tiempo promedio de espera en cola para los
        # agentes de soporte durante esta corrida del modelo
        self.mean_queue_time_support_agent = 0

    # Función generadora que representa el generador DES para las llegadas
    # de clientes
    def generator_customer_arrivals(self):
        # Usamos un bucle infinito para seguir haciendo esto indefinidamente mientras
        # la simulación esté en ejecución
        while True:
            # Incrementar el contador de clientes en 1 (esto significa que nuestro primer
            # cliente tendrá un ID de 1)
            self.customer_counter += 1

            # Crear un nuevo cliente: una instancia de la clase Customer que
            # definimos arriba. Recuerda que pasamos el ID al crear un
            # cliente, así que aquí pasamos el contador de clientes como ID.
            c = Customer(self.customer_counter)

            # Decirle a SimPy que inicie la función generadora use_customer_service_helpline
            # con este cliente (la función generadora que modelará el recorrido
            # del cliente a través del sistema)
            self.env.process(self.use_customer_service_helpline (c))

            # Muestrear aleatoriamente el tiempo hasta la llegada del siguiente cliente.
            # Aquí muestreamos de una distribución exponencial (común para tiempos
            # entre llegadas) y pasamos un valor lambda de 1 / media. La media del
            # tiempo entre llegadas está almacenada en la clase glo.
            sampled_inter_arrival_time = random.expovariate(1.0 / glo.time_units_between_customer_arrivals)

            # Congelar esta instancia de esta función hasta que transcurra el tiempo
            # entre llegadas que muestreamos arriba. Nota: el tiempo en SimPy progresa
            # en “unidades de tiempo”, que pueden representar lo que quieras
            # (solo asegúrate de ser consistente dentro del modelo)
            yield self.env.timeout(sampled_inter_arrival_time)

    # Función generadora que representa la ruta de un cliente que llama a nuestra línea de ayuda.
    # Aquí la ruta es extremadamente simple: un cliente
    # llega al sistema de llamadas, espera a ser conectado con un agente de soporte,
    # pasa un tiempo variable siendo atendido por el agente y luego se va,
    # liberando al agente para ayudar a la siguiente persona.
    # El objeto cliente se pasa a la función generadora para poder
    # extraer/registrar información en él.
    def use_customer_service_helpline(self, customer):
        # Registrar el momento en que el cliente comenzó a hacer cola para un agente de soporte
        start_q_customer_support_agent = self.env.now

        # Este código solicita un recurso de agente de soporte al cliente y realiza todo el
        # bloque siguiente con ese recurso mantenido (y por lo tanto
        # no utilizable por otro cliente)
        with self.customer_support_agent.request() as req:
            # Congelar la función hasta que se pueda satisfacer la solicitud del agente de soporte.
            # El cliente está actualmente en cola.
            yield req

            # Cuando llegamos a esta parte del código, el control ha vuelto a
            # la funci


### Clase Trial

Nuestra clase final es **Trial**. Representa un **conjunto (lote)** de corridas del modelo. Se encarga de configurar y lanzar todas las corridas, así como de **almacenar, registrar y mostrar** los resultados del experimento.

#### Ejemplo de clase Trial


In [4]:
class Trial:
    # El constructor configura un DataFrame de pandas que almacenará los
    # resultados clave de cada corrida (aquí solo el tiempo promedio de cola
    # para el agente de soporte al cliente), asociados al número de corrida,
    # con el número de corrida como índice.
    def  __init__(self):
        self.df_trial_results = pd.DataFrame()
        self.df_trial_results["Run Number"] = [0]
        self.df_trial_results["Mean Queue Time Customer Supoprt Agent"] = [0.0]
        self.df_trial_results.set_index("Run Number", inplace=True)

    # Método para imprimir los resultados del experimento. En modelos del mundo real,
    # probablemente también los guardarías (o los guardarías en lugar de imprimirlos).
    def print_trial_results(self):
        print ("Trial Results")
        print (self.df_trial_results)

    # Método para ejecutar un experimento
    def run_trial(self):
        # Ejecutar la simulación por el número de corridas especificado en la clase global.
        # Para cada corrida, creamos una nueva instancia de la clase Model y llamamos
        # a su método run, lo cual pone todo en marcha. Una vez que la corrida
        # termina, tomamos los resultados almacenados (aquí solo el tiempo promedio
        # de cola) y los guardamos asociados al número de corrida en el DataFrame
        # de resultados del experimento.
        for run in range(glo.number_of_runs):
            my_model = Model(run)
            my_model.run()

            self.df_trial_results.loc[run] = [my_model.mean_queue_time_support_agent]

        # Cuando el experimento (es decir, todas las corridas) haya finalizado, imprime
        # los resultados finales
        self.print_trial_results()



Por supuesto, luego podemos **calcular promedios** sobre las corridas del experimento … aunque probablemente deberíamos hacerlo en un **método aparte** dentro de la clase **Trial**.


## Un Ejemplo de lo anterior

En el ejemplo que vamos a revisar, modelaremos un sistema muy sencillo: pacientes que llegan a una clínica para una consulta con la enfermera.
Un tipo de entidad, un generador, una actividad, una cola, un sumidero (*sink*), un tipo de recurso.

<p align="center">
  <img src="images/example_simplest_model.png" alt="Modelo simple" width="800"/>
</p>

Un modelo en **SimPy** puede parecer bastante complejo al principio, especialmente para un modelo tan simple como este. Pero la buena noticia es que la estructura general siempre es la misma, sin importar la complejidad.

## importación

Primero necesitamos importar las librerías, variarán según el modelo a desarrollar y lo que necesites, pero estas tres probablemente siempre estarán ahí (¡la primera es obligatoria!).


In [5]:
import simpy
import random
import pandas as pd

**random** nos da acceso al muestreo estocástico a partir de distribuciones de probabilidad

### Clase glo

Recuerda: la clase **glo** almacena nuestros valores de parámetros globales para el modelo, de modo que podamos cambiar fácilmente aspectos del mismo para probar diferentes escenarios.


In [6]:
# Clase para almacenar valores de parámetros globales. No creamos una instancia de esta
# clase, simplemente accedemos a los números que contiene.
class glo:
    patient_inter = 5
    mean_n_consult_time = 6
    number_of_nurses = 1
    sim_duration = 120
    number_of_runs = 5


### Clase **patient** (Entidad)

In [7]:
# Clase que representa a los pacientes que llegan a la clínica. Aquí, los pacientes
# tienen dos atributos que llevan consigo: su ID (cedula) y el tiempo que pasaron
# haciendo cola por la enfermera. El ID se pasa al crear un nuevo paciente.
class Patient:
    def __init__(self, p_id):
        self.id = p_id
        self.q_time_nurse = 0


### Clase **Model**

In [8]:
# Clase que representa nuestro modelo de la clínica.
class Model:
    # Constructor para preparar el modelo para una corrida. Pasamos un número de
    # corrida cuando creamos un nuevo modelo.
    def __init__(self, run_number):
        # Crear un entorno de SimPy en el que vivirá todo
        self.env = simpy.Environment()

        # Crear un contador de pacientes (que usaremos como ID de paciente)
        self.patient_counter = 0

        # Crear un recurso de SimPy para representar a una enfermera, que vivirá en el
        # entorno creado arriba. La cantidad de este recurso que tenemos está
        # especificada por la capacidad, y tomamos este valor de nuestra clase glo.
        self.nurse = simpy.Resource(self.env, capacity=glo.number_of_nurses)

        # Guardar el número de corrida recibido
        self.run_number = run_number

        # Crear un nuevo DataFrame de Pandas que almacenará algunos resultados
        # asociados al ID del paciente (que usaremos como índice).
        self.results_df = pd.DataFrame()
        self.results_df["Patient ID"] = [1]
        self.results_df["Q Time Nurse"] = [0.0]
        self.results_df["Time with Nurse"] = [0.0]
        self.results_df.set_index("Patient ID", inplace=True)

        # Crear un atributo para almacenar el tiempo promedio en cola para la enfermera
        # durante esta corrida del modelo
        self.mean_q_time_nurse = 0

    # Función generadora que representa el generador DES para las
    # llegadas de pacientes
    def generator_patient_arrivals(self):
        # Usamos un bucle infinito aquí para seguir haciendo esto indefinidamente
        # mientras la simulación se ejecuta
        while True:
            # Incrementar el contador de pacientes en 1 (esto significa que nuestro primer
            # paciente tendrá un ID de 1)
            self.patient_counter += 1

            # Crear un nuevo paciente: una instancia de la clase Patient que
            # definimos arriba. Recuerda, pasamos el ID al crear un
            # paciente; aquí pasamos el contador de pacientes para usarlo como ID.
            p = Patient(self.patient_counter)

            # Indicar a SimPy que inicie la función generadora attend_clinic con
            # este paciente (la función generadora que modelará el
            # recorrido del paciente a través del sistema)
            self.env.process(self.attend_clinic(p))

            # Muestrear aleatoriamente el tiempo hasta la llegada del siguiente paciente.
            # Aquí muestreamos de una distribución exponencial (común para tiempos
            # entre llegadas) y pasamos un valor lambda de 1 / media. La media del
            # tiempo entre llegadas está almacenada en la clase glo.
            sampled_inter = random.expovariate(1.0 / glo.patient_inter)

            # Congelar esta instancia de esta función hasta que haya transcurrido el
            # tiempo entre llegadas que muestreamos arriba. Nota: el tiempo en
            # SimPy progresa en “Unidades de Tiempo”, que pueden representar lo que
            # quieras (solo asegúrate de ser consistente dentro del modelo)
            yield self.env.timeout(sampled_inter)

    # Función generadora que representa la ruta de un paciente al pasar
    # por la clínica. Aquí la ruta es extremadamente simple: un paciente
    # llega, espera para ver a una enfermera y luego se va.
    # El objeto patient se pasa a la función generadora para poder
    # extraer/registrar información en él
    def attend_clinic(self, patient):
        # Registrar el momento en que el paciente comenzó a hacer cola para una enfermera
        start_q_nurse = self.env.now

        # Este código solicita un recurso de enfermera y ejecuta todo el
        # bloque siguiente con ese recurso retenido (y por lo tanto
        # no utilizable por otro paciente)
        with self.nurse.request() as req:
            # Congelar la función hasta que se pueda satisfacer la solicitud de una enfermera.
            # El paciente está actualmente en cola.
            yield req

            # Cuando llegamos a esta parte del código, el control ha vuelto a
            # la función generadora y, por lo tanto, la solicitud de una enfermera
            # se ha satisfecho. Ahora tenemos a la enfermera y hemos dejado de hacer cola,
            # así que podemos registrar el momento actual como el fin de la cola.
            end_q_nurse = self.env.now

            # Calcular el tiempo que este paciente estuvo en cola para la enfermera y
            # registrarlo en el atributo correspondiente del paciente.
            patient.q_time_nurse = end_q_nurse - start_q_nurse

            # Ahora muestrearemos aleatoriamente el tiempo que este paciente pasará con
            # la enfermera. Aquí usamos una distribución Exponencial por simplicidad,
            # pero típicamente usarías una Log-Normal en un modelo real
            # (volveremos a eso). Como al muestrear los tiempos entre llegadas,
            # tomamos la media de la clase g y pasamos 1 / media
            # como el valor lambda.
            sampled_nurse_act_time = random.expovariate(1.0 /
                                                        glo.mean_n_consult_time)

            # Aquí almacenaremos el tiempo en cola para la enfermera y el tiempo
            # muestreado para pasar con la enfermera en el DataFrame de resultados
            # asociado al ID de este paciente. En modelos del mundo real,
            # puede que no quieras molestarte en almacenar los tiempos de actividad
            # muestreados; pero como este es un modelo simple, lo haremos aquí.
            # Usamos una propiedad útil de pandas llamada .at, que funciona un poco
            # como .loc. .at nos permite acceder (y por tanto cambiar) una
            # celda particular en nuestro DataFrame proporcionando la fila y la columna.
            # Aquí especificamos la fila como el ID del paciente (el índice) y la
            # columna del valor que queremos actualizar para ese paciente.
            self.results_df.at[patient.id, "Q Time Nurse"] = (
                patient.q_time_nurse)
            self.results_df.at[patient.id, "Time with Nurse"] = (
                sampled_nurse_act_time)

            # Congelar esta función en el lugar durante el tiempo de actividad muestreado
            # arriba. Esto representa al paciente pasando tiempo con la
            # enfermera.
            yield self.env.timeout(sampled_nurse_act_time)

            # Cuando el tiempo anterior transcurra, la función generadora volverá
            # aquí. Como no hemos escrito nada más, la función
            # simplemente terminará. Este es un sumidero (sink). Podríamos elegir
            # agregar algo aquí si quisiéramos registrar algo, p. ej., un contador
            # del número de pacientes que salieron, o registrar algo sobre los
            # pacientes que salieron en un sumidero particular, etc.

    # Este método calcula resultados sobre una sola corrida. Aquí solo calculamos
    # una media, pero en modelos del mundo real probablemente querrás calcular más.
    def calculate_run_results(self):
        # Tomar la media de los tiempos de espera en cola para la enfermera entre los
        # pacientes en esta corrida del modelo.
        self.mean_q_time_nurse = self.results_df["Q Time Nurse"].mean()

    # El método run inicia los generadores DES de entidades, ejecuta la simulación
    # y a su vez llama a lo que necesitemos para generar resultados de la corrida
    def run(self):
        # Iniciar nuestros generadores DES de entidades que crean nuevos pacientes.
        # Solo tenemos uno en este modelo, pero necesitaríamos hacer esto para cada
        # generador si hubiera múltiples.
        self.env.process(self.generator_patient_arrivals())

        # Ejecutar el modelo por la duración especificada en la clase g
        self.env.run(until=glo.sim_duration)

        # Ahora que la corrida de la simulación ha terminado, llamar al método que
        # calcula los resultados de la corrida
        self.calculate_run_results()

        # Imprimir el número de corrida junto con los resultados a nivel de paciente
        # de esta corrida del modelo
        print (f"Run Number {self.run_number}")
        print (self.results_df)


Ahora solo necesitamos ejecutar el experimento (**trial**) e imprimir los resultados

In [11]:
class Trial:
    # El constructor configura un DataFrame de pandas que almacenará los
    # resultados clave de cada corrida (aquí solo el tiempo promedio en cola
    # para la enfermera), asociados al número de corrida, con el número de
    # corrida como índice.
    def  __init__(self):
        self.df_trial_results = pd.DataFrame()
        self.df_trial_results["Run Number"] = [0]
        self.df_trial_results["Mean Q Time Nurse"] = [0.0]
        self.df_trial_results.set_index("Run Number", inplace=True)

    # Método para imprimir los resultados del experimento. En modelos del
    # mundo real, probablemente también los guardarías (o los guardarías en
    # lugar de imprimirlos).
    def print_trial_results(self):
        print ("Trial Results")
        print (self.df_trial_results)

    # Método para ejecutar un experimento
    def run_trial(self):
        # Ejecutar la simulación por el número de corridas especificado en la
        # clase g. Para cada corrida, creamos una nueva instancia de la clase
        # Model y llamamos a su método run, lo que pone todo en marcha. Una vez
        # que la corrida haya terminado, tomamos los resultados almacenados
        # (aquí solo el tiempo promedio en cola) y los guardamos asociados al
        # número de corrida en el DataFrame de resultados del experimento.
        for run in range(glo.number_of_runs):
            my_model = Model(run)
            my_model.run()

            self.df_trial_results.loc[run] = [my_model.mean_q_time_nurse]

        # Cuando el experimento (es decir, todas las corridas) haya finalizado,
        # imprime los resultados finales
        self.print_trial_results()


In [12]:
# Crear una instancia de la clase Trial
my_trial = Trial()

# Llamar al método run_trial de nuestro objeto Trial
my_trial.run_trial()


Run Number 0
            Q Time Nurse  Time with Nurse
Patient ID                               
1               0.000000         1.761805
2               0.000000         0.007758
3               0.000000         1.313114
4               0.000000         2.191065
5               0.000000        11.363144
6               9.155045         0.873748
7               5.646492         7.926761
8               9.131475         1.018965
9               9.845369         5.394793
10             11.737359         2.246780
11              8.109080         8.640733
12             15.305926         2.421639
13             17.402833         0.716707
14             13.874227         6.638245
15             11.974415         4.658438
16             12.957290         1.033603
17             13.685314         3.101963
18             14.847407         4.945269
19             14.445955         3.514122
20             17.860344         2.332753
21             19.606898        10.101573
22             28.115